<a href="https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
#setup cell
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shahzad-jatoi/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / ranking.** Given a page's features, I want to output a continuous priority score (not a hard class) that ranks pages by how worthwhile they are to refresh next. This is closer to ranking than plain classification because the actual deliverable a content team needs isn't "declining vs. not declining" it's an ordered queue: which 20 pages do we fix first given limited editorial hours. A binary label would throw away exactly the information (relative priority) the action depends on.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy target:** `is_declining_label`, defined as `trend_direction == "down"` (the same proxy the starter pipeline already uses). This is a *defined rule*, not a directly observed outcome "declining" is a bucketed simplification of the real underlying signal (`trend_pct`), so it's a proxy for the harder-to-define concept of "worth refreshing." I'm intentionally not using `trend_pct` itself as a feature, since it's what the label is derived from and would leak the answer into the model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K** (specifically Precision@50), matching what the starter pipeline already reports. It directly matches the real action: a content team only has bandwidth to refresh a fixed number of pages per cycle, so what matters isn't overall accuracy across all 30,000 pages it's "of the top K pages my ranking flags, how many are actually worth acting on." I'll compare this against the hand-written rule baseline (stale × visible) to defend that any added complexity is earning its keep.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#code cell

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build the proxy label (same rule the starter pipeline uses)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# The unit of analysis: one row = one page
lane_slice = df[[
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"
]]

print(f"Rows: {lane_slice.shape[0]}  (one row = one page)")
print(f"Declining rate: {lane_slice['is_declining_label'].mean():.3f}")
lane_slice.head(5)

Rows: 30000  (one row = one page)
Declining rate: 0.542


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,down,1
1,445,25,15320,20.3,0.05,2481.0,down,1
2,141,20,12581,36.5,0.09,3515.0,down,1
3,463,22,11751,6.2,0.49,NaN,stable,0
4,263,14,19140,44.0,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "stale AND visible" only checks two conditions and treats every page that clears them as equally worth refreshing. But the real signal is a messier combination: `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `content_age_days`, and `word_count` likely interact in non-obvious ways  e.g. a stale page with very low impressions might not be worth touching at all, while a moderately-stale page in a strong position tier with collapsing CTR could be a high-value fix. A hand-written if-statement can encode maybe two or three conditions before it becomes unreadable; a shallow decision tree or similar model can weigh six signals at once and rank by degree, not just pass/fail, which is exactly what a priority queue needs. Notebook 02 already showed this concretely: the hand rule and a depth-2 tree don't always agree on the top pages, and the disagreement itself is informative.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.